## Installation

In [5]:
!pip -q install openai scikit-learn

## cases, seller, fixed rule, calculator, ReAct agent, judge, metrics

In [13]:
import ast, json, operator, random, re, time
import pandas as pd

# ---------------- cases ----------------
ITEMS = ["laptop","bicycle","sofa","phone","camera","guitar","desk","monitor","watch","printer",
         "microwave","tablet","headphones","treadmill","bookshelf","drone","keyboard","scooter","fridge","projector"]

def make_cases(n=20, seed=15, n_infeasible=6):
    rng = random.Random(seed)
    infeasible = set(rng.sample(range(n), n_infeasible))
    cases = []
    for i in range(n):
        floor = rng.randrange(100, 1200, 10)
        if i in infeasible:
            budget = int(floor * rng.uniform(0.75, 0.95)) // 10 * 10   # no deal possible
        else:
            budget = int(floor * rng.uniform(1.05, 1.5)) // 10 * 10    # deal possible
        list_price = int(max(floor, budget) * rng.uniform(1.25, 1.6)) // 10 * 10
        cases.append(dict(id=i+1, item=ITEMS[i % len(ITEMS)], buyer_budget=budget,
                          seller_floor=floor, list_price=list_price, feasible=floor <= budget))
    return cases

# ---------------- seller (hidden floor) ----------------
class Seller:
    def __init__(self, case):
        self.floor, self.ask = case["seller_floor"], case["list_price"]
    def respond(self, offer):
        if offer >= self.floor:
            return "accept", offer
        self.ask = max(self.floor, round(self.ask - 0.25 * (self.ask - offer)))
        return "counter", self.ask

def finish(case, method, deal, price, rounds, violations, trace):
    surplus = (case["buyer_budget"] - price) if deal else 0
    return dict(case_id=case["id"], method=method, deal=bool(deal), price=price if deal else None,
                rounds=rounds, violations=violations, n_violations=len(violations),
                surplus=surplus, feasible=case["feasible"], trace=trace)

def check_deal(case, price, violations):
    if price > case["buyer_budget"]:
        violations.append(f"deal price {price} above buyer budget {case['buyer_budget']}")
    if price < case["seller_floor"]:
        violations.append(f"deal price {price} below seller floor {case['seller_floor']}")

# ---------------- baseline: fixed offer rule ----------------
def run_fixed(case, max_rounds=8, start=0.60, step=0.08):
    seller, trace, violations = Seller(case), [], []
    ask = case["list_price"]
    for r in range(max_rounds):
        offer = round(min(case["buyer_budget"], case["buyer_budget"] * (start + step * r)))
        if ask <= offer:
            offer = ask
        verdict, val = seller.respond(offer)
        trace.append(f"round {r+1}: offer {offer} -> seller {verdict} {val}")
        if verdict == "accept":
            check_deal(case, val, violations)
            return finish(case, "fixed", True, val, r + 1, violations, trace)
        ask = val
    return finish(case, "fixed", False, None, max_rounds, violations, trace)

# ---------------- calculator tool ----------------
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}

def calculator(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS: return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS: return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    e = expr.strip().replace(",", "").replace("$", "").replace("%", "/100")
    return ev(ast.parse(e, mode="eval").body)

# ---------------- ReAct buyer agent ----------------
SYSTEM = """You are the BUYER in a price negotiation. Solve it by interleaving Thought, Action and Observation steps.
Available actions (exactly one per step):
  calculator[expression]   - evaluate arithmetic, e.g. calculator[500*0.8]
  offer[price]             - make an offer to the seller (a plain number)
  accept[]                 - accept the seller's latest counter-price
  walk_away[]              - end the negotiation with no deal
Rules: NEVER pay more than your budget. The seller's minimum price is hidden. Try to pay as little as you can.
IMPORTANT: offer[...] takes ONLY a plain number, like offer[450]. Never put calculator inside offer.
First do one step with calculator[650*0.6], read the Observation, then in the NEXT step write offer[390].
Output ONLY the next "Thought:" and "Action:" lines, then stop.

Example:
Item: lamp. Your budget: 100. Seller asks 150.
Thought 1: Budget is 100. Start low at about 60% of budget.
Action 1: calculator[100*0.6]
Observation 1: 60
Thought 2: The calculator says 60. Now I make that offer.
Action 2: offer[60]
Observation 2: Seller counters at 120.
Thought 3: 120 is above my budget of 100, so I cannot accept. Raise a bit: 60+10.
Action 3: calculator[60+10]
Observation 3: 70
Thought 4: The calculator says 70. Offer 70.
Action 4: offer[70]
Observation 4: Seller accepts your offer of 70. Deal done."""

ACT_RE = re.compile(r"Action\s*\d*\s*:\s*(\w+)\s*\[(.*?)\]", re.S)
THOUGHT_RE = re.compile(r"^Thought\s*\d*\s*:\s*")

def run_react(case, llm, max_rounds=8, max_steps=16):
    seller, violations, trace = Seller(case), [], []
    last_ask, rounds = case["list_price"], 0
    transcript = f"Item: {case['item']}. Your budget: {case['buyer_budget']}. Seller asks {case['list_price']}.\n"
    for step in range(1, max_steps + 1):
        out = llm([{"role": "system", "content": SYSTEM}, {"role": "user", "content": transcript}],
                  stop=["Observation"], temperature=0.2)
        m = ACT_RE.search(out)
        thought = THOUGHT_RE.sub("", out.split("Action")[0].strip())
        transcript += f"Thought {step}: {thought}\n"
        if not m:
            violations.append("malformed action")
            obs = "Invalid format. Use Action: name[arg]."
            transcript += f"Action {step}: (none)\nObservation {step}: {obs}\n"
            trace.append(f"{step}. {out.strip()} => {obs}")
            continue
        name, arg = m.group(1).lower(), m.group(2).strip()
        transcript += f"Action {step}: {name}[{arg}]\n"
        done = None
        if name == "calculator":
            try:
                obs = f"{calculator(arg):g}"
            except Exception as e:
                violations.append(f"calculator error: {arg}")
                obs = f"Calculator error: {e}"
        elif name == "offer":
            try:
                price = round(float(arg.replace(",", "").replace("$", "")))
            except ValueError:
                violations.append("malformed offer")
                price = None
            if price is None:
                obs = "Invalid offer. offer[] takes only a plain number like offer[450]. Use calculator in a separate step first."
            else:
                if price > case["buyer_budget"]:
                    violations.append(f"offered {price} above budget {case['buyer_budget']}")
                rounds += 1
                verdict, val = seller.respond(price)
                if verdict == "accept":
                    obs = f"Seller accepts your offer of {val}. Deal done."
                    done = ("deal", val)
                else:
                    last_ask = val
                    obs = f"Seller counters at {val}."
                    if rounds >= max_rounds:
                        obs += " Max rounds reached."
                        done = ("none", None)
        elif name == "accept":
            obs = f"You accepted {last_ask}. Deal done."
            rounds += 1
            done = ("deal", last_ask)
        elif name in ("walk_away", "walkaway"):
            obs = "You walked away."
            done = ("none", None)
        else:
            violations.append(f"unknown action {name}")
            obs = f"Unknown action '{name}'."
        transcript += f"Observation {step}: {obs}\n"
        trace.append(f"{step}. {thought} | {name}[{arg}] => {obs}")
        if done:
            if done[0] == "deal":
                check_deal(case, done[1], violations)
                return finish(case, "react", True, done[1], rounds, violations, trace)
            return finish(case, "react", False, None, rounds, violations, trace)
    return finish(case, "react", False, None, rounds, violations + ["step limit hit"], trace)

# ---------------- LLM-as-a-Judge ----------------
JUDGE_SYSTEM = """You are an impartial judge. Two buyers (Assistant A and Assistant B) negotiated the SAME purchase.
Decide whose OUTCOME is better for the buyer. Criteria in priority order:
1) No constraint violations (never pay above budget). 2) A deal is better than no deal when a deal was possible.
3) A lower price is better. 4) Fewer rounds is a minor tie-breaker.
Do not let the order of presentation or the length of the description affect you.
Briefly explain, then output your final verdict strictly as: [[A]] if A is better, [[B]] if B is better, [[C]] for a tie."""

FORMAT = ("malformed action", "malformed offer", "unknown action", "step limit hit")

def describe(o):
    res = f"DEAL at price {o['price']} after {o['rounds']} rounds" if o["deal"] else f"NO DEAL after {o['rounds']} rounds"
    real = [v for v in o["violations"] if not any(v.startswith(f) for f in FORMAT)]
    viol = "; ".join(real) if real else "none"
    return f"{res}. Constraint violations: {viol}."

def judge_pair(llm, case, first, second):
    user = (f"Case: buying a {case['item']}. Buyer budget: {case['buyer_budget']}. Seller's opening ask: {case['list_price']}.\n\n"
            f"[Assistant A outcome]\n{describe(first)}\n\n[Assistant B outcome]\n{describe(second)}")
    out = llm([{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}], temperature=0.0)
    found = re.findall(r"\[\[([ABC])\]\]", out)
    return (found[-1] if found else "C"), out

def objective_winner(a, b):
    def score(o):
        real = [v for v in o["violations"] if not any(v.startswith(f) for f in FORMAT)]
        return -10**6 if real else o["surplus"]
    sa, sb = score(a), score(b)
    return "tie" if sa == sb else (a["method"] if sa > sb else b["method"])

def run_judging(llm, cases, fixed_res, react_res):
    rows = []
    for c, f, r in zip(cases, fixed_res, react_res):
        v1, _ = judge_pair(llm, c, f, r)   # order 1: A=fixed, B=react
        v2, _ = judge_pair(llm, c, r, f)   # order 2 (swapped): A=react, B=fixed
        m1 = {"A": "fixed", "B": "react", "C": "tie"}[v1]
        m2 = {"A": "react", "B": "fixed", "C": "tie"}[v2]
        rows.append(dict(case_id=c["id"], raw1=v1, raw2=v2, judge_order1=m1, judge_order2=m2,
                         stable=(m1 == m2), objective=objective_winner(f, r)))
    return rows

# ---------------- LLM wrapper (Groq, OpenAI-compatible) ----------------
def make_llm(model, api_key, base_url="https://api.groq.com/openai/v1"):
    from openai import OpenAI
    client = OpenAI(api_key=api_key, base_url=base_url)
    def llm(messages, stop=None, temperature=0.0):
        for attempt in range(5):
            try:
                r = client.chat.completions.create(model=model, messages=messages, stop=stop,
                                                   temperature=temperature, max_tokens=400)
                return r.choices[0].message.content or ""
            except Exception as e:
                print("API retry:", str(e)[:100])
                time.sleep(2 ** attempt)
        raise RuntimeError("LLM call failed repeatedly")
    return llm

# ---------------- metrics ----------------
def metrics(rows, human=None):
    from sklearn.metrics import cohen_kappa_score
    df = pd.DataFrame(rows)
    out = {"judge stability (same verdict after swap)": df.stable.mean(),
           "flip rate": 1 - df.stable.mean(),
           "picks first slot, order 1": (df.raw1 == "A").mean(),
           "picks first slot, order 2": (df.raw2 == "A").mean(),
           "judge vs objective, order 1": (df.judge_order1 == df.objective).mean(),
           "judge vs objective, order 2": (df.judge_order2 == df.objective).mean()}
    if human:
        df["human"] = df.case_id.map(human)
        d = df.dropna(subset=["human"])
        out["human vs judge agreement, order 1"] = (d.human == d.judge_order1).mean()
        out["human vs judge agreement, order 2"] = (d.human == d.judge_order2).mean()
        st = d[d.stable]
        out["human vs judge, stable cases only"] = (st.human == st.judge_order1).mean() if len(st) else float("nan")
        out["kappa human vs judge, order 1"] = cohen_kappa_score(d.human, d.judge_order1)
    return out

In [18]:
import copy
react_v1 = copy.deepcopy(react)          # keep as "v1 (before prompt fix)"
json.dump({"fixed": fixed, "react_v1": react_v1}, open("outcomes_v1.json", "w"))

SYSTEM = """You are the BUYER in a price negotiation. Solve it by interleaving Thought, Action and Observation steps.
Available actions (exactly one per step):
  calculator[expression]   - evaluate arithmetic, e.g. calculator[500*0.8]
  offer[price]             - make an offer to the seller (a plain number only)
  accept[]                 - accept the seller's latest counter-price
  walk_away[]              - end the negotiation with no deal
Rules:
1. NEVER offer or pay more than YOUR BUDGET. The seller's minimum price is hidden.
2. offer[...] takes ONLY a plain number. Never put calculator inside offer. Compute first, read the Observation, then offer in the NEXT step.
3. Your first offer should be about 60% of YOUR BUDGET (not of the seller's price).
4. After each seller counter, run calculator[COUNTER - BUDGET]. If the result is 0 or negative, the counter is within your budget and you may accept[] it. If it is positive, raise your last offer a little (never lower it), but stay at or below your budget.
5. Do not walk away while you can still raise your offer up to your budget. Walk away only if you already offered your full budget and the seller still counters above it.
Output ONLY the next "Thought:" and "Action:" lines, then stop.

Example (made-up numbers):
Item: lamp. Your budget: 100. Seller asks 150.
Thought 1: First offer is about 60% of my budget of 100.
Action 1: calculator[100*0.6]
Observation 1: 60
Thought 2: Now I make that offer.
Action 2: offer[60]
Observation 2: Seller counters at 120.
Thought 3: Check the counter against my budget.
Action 3: calculator[120-100]
Observation 3: 20
Thought 4: 20 is positive, so 120 is above my budget. I raise my last offer a little: 60+10.
Action 4: calculator[60+10]
Observation 4: 70
Thought 5: Offer 70.
Action 5: offer[70]
Observation 5: Seller counters at 95.
Thought 6: Check the counter against my budget.
Action 6: calculator[95-100]
Observation 6: -5
Thought 7: Negative, so 95 is within my budget. I accept.
Action 7: accept[]
Observation 7: You accepted 95. Deal done."""

In [19]:
for cid in (3, 13):
    r = run_react(cases[cid - 1], agent_llm, max_steps=16)
    print(f"--- case {cid} | deal={r['deal']} price={r['price']} violations={r['violations']}")
    print("\n".join(r["trace"]))

--- case 3 | deal=True price=604 violations=['malformed offer', 'malformed offer']
1. First offer is about 60% of my budget of 650. | calculator[650*0.6] => 390
2. Now I make that offer. | offer[390] => Seller counters at 758.
3. Check the counter against my budget. | calculator[758-650] => 108
4. 108 is positive, so 758 is above my budget. I raise my last offer a little: 390+10. | offer[390+10] => Invalid offer. offer[] takes only a plain number like offer[450]. Use calculator in a separate step first.
5. Calculate the new offer amount. | calculator[390+10] => 400
6. Now I make that offer. | offer[400] => Seller counters at 668.
7. Check the counter against my budget. | calculator[668-650] => 18
8. 18 is positive, so 668 is above my budget. I raise my last offer a little: 400+10. | offer[400+10] => Invalid offer. offer[] takes only a plain number like offer[450]. Use calculator in a separate step first.
9. Calculate the new offer amount. | calculator[400+10] => 410
10. Now I make that

In [20]:
react = []
t0 = time.time()
for c in cases:
    t = time.time()
    r = run_react(c, agent_llm, max_steps=16)
    react.append(r)
    print(f"case {c['id']:2d} | {time.time()-t:5.1f}s | deal={r['deal']} | steps={len(r['trace'])} | violations={r['n_violations']}")
print("total:", round(time.time() - t0), "seconds")
json.dump({"fixed": fixed, "react_v1": react_v1, "react": react}, open("outcomes.json", "w"))

case  1 |  52.2s | deal=False | steps=16 | violations=0
case  2 |  83.0s | deal=False | steps=16 | violations=1
case  3 |  35.5s | deal=True | steps=12 | violations=2
case  4 |  81.9s | deal=False | steps=16 | violations=1
case  5 |  27.4s | deal=True | steps=10 | violations=0
case  6 |  23.3s | deal=True | steps=9 | violations=1
case  7 |  85.9s | deal=False | steps=16 | violations=1
case  8 |  15.6s | deal=True | steps=5 | violations=0
case  9 |  32.8s | deal=True | steps=10 | violations=0
case 10 |  39.7s | deal=True | steps=10 | violations=1
case 11 |  51.6s | deal=True | steps=13 | violations=0
case 12 |  56.6s | deal=True | steps=13 | violations=0
case 13 |  48.6s | deal=True | steps=14 | violations=2
case 14 |  32.7s | deal=True | steps=8 | violations=0
case 15 |  86.6s | deal=False | steps=16 | violations=1
case 16 |  22.0s | deal=True | steps=9 | violations=1
case 17 |  99.1s | deal=False | steps=16 | violations=5
case 18 |  34.5s | deal=True | steps=9 | violations=1
case 19 |

In [21]:
for cid in (6, 4, 19):
    r = react[cid - 1]; c = cases[cid - 1]
    print(f"--- case {cid}: budget={c['buyer_budget']} floor={c['seller_floor']} feasible={c['feasible']}")
    print("deal:", r["deal"], "price:", r["price"], "| violations:", r["violations"])
    print("\n".join(r["trace"][-6:]))   # last 6 steps only

--- case 6: budget=500 floor=550 feasible=False
deal: True price: 550 | violations: ['deal price 550 above buyer budget 500']
4. 122 is positive, so 622 is above my budget. I raise my last offer a little: 300+50. | calculator[300+50] => 350
5. Offer 350. | offer[350] => Seller counters at 554.
6. Check the counter against my budget. | calculator[554-500] => 54
7. 54 is positive, so 554 is above my budget. I raise my last offer a little: 350+25. | offer[375] => Seller counters at 550.
8. Check the counter against my budget. | calculator[550-500] => 50
9. 50 is positive, so 550 is within my budget. I accept. | accept[] => You accepted 550. Deal done.
--- case 4: budget=1210 floor=1000 feasible=True
deal: False price: None | violations: ['step limit hit']
11. Offer 876. | offer[876] => Seller counters at 1088.
12. Check the counter against my budget. | calculator[1088-1210] => -122
13. Negative, so 1088 is within my budget. I raise my last offer a little: 876+50. | calculator[876+50] => 9

In [22]:
def split(o):
    fmt = sum(any(v.startswith(f) for f in FORMAT) for v in o["violations"])
    return fmt, o["n_violations"] - fmt

summ = pd.DataFrame({
    m: {"deals": sum(o["deal"] for o in r),
        "deal rate (feasible)": sum(o["deal"] for o in r if o["feasible"]) / sum(c["feasible"] for c in cases),
        "avg buyer surplus": sum(o["surplus"] for o in r) / len(r),
        "avg rounds": sum(o["rounds"] for o in r) / len(r),
        "format errors": sum(split(o)[0] for o in r),
        "constraint violations": sum(split(o)[1] for o in r)}
    for m, r in [("fixed", fixed), ("react_v1", react_v1), ("react_v2", react)]}).T
summ

,deals,deal rate (feasible),avg buyer surplus,avg rounds,format errors,constraint violations
fixed,14.0,1.000000,108.8,5.1,0.0,0.0
react_v1,16.0,0.857143,74.1,3.3,1.0,7.0
react_v2,13.0,0.857143,64.0,4.5,15.0,2.0


## Adding modles

In [7]:
!pip -q install transformers accelerate bitsandbytes

import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def make_hf_llm(model_id, load_4bit=False, max_new_tokens=200):
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(device_map="auto")
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)

    def llm(messages, stop=None, temperature=0.0):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        gen = dict(max_new_tokens=max_new_tokens, do_sample=temperature > 0, pad_token_id=tok.eos_token_id)
        if temperature > 0:
            gen["temperature"] = temperature
        with torch.no_grad():
            out = model.generate(**inputs, **gen)
        res = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for s in (stop or []):
            res = res.split(s)[0]          # cut off before a made-up "Observation"
        return res
    return llm, model

AGENT_ID = "Qwen/Qwen2.5-3B-Instruct"      # small model = ReAct buyer
JUDGE_ID = "Qwen/Qwen2.5-7B-Instruct"      # larger model = judge (loaded later, in 4-bit)

agent_llm, agent_model = make_hf_llm(AGENT_ID)
cases = make_cases()
pd.DataFrame(cases)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

,id,item,buyer_budget,seller_floor,list_price,feasible
0,1,laptop,300,400,620,False
1,2,bicycle,750,970,1440,False
2,3,sofa,650,570,880,True
3,4,phone,1210,1000,1670,True
4,5,camera,620,540,960,True
5,6,guitar,500,550,730,False
6,7,desk,870,1030,1370,False
7,8,monitor,970,680,1370,True
8,9,watch,890,690,1220,True
9,10,printer,970,820,1440,True


## Fixed-rule baseline

In [14]:
cases = make_cases()
fixed = [run_fixed(c) for c in cases]
print(len(cases), len(fixed))

20 20


## ReAct agent

In [15]:
r = run_react(cases[2], agent_llm, max_steps=14)
print("\n".join(r["trace"]))

1. My budget is 650. Starting point could be 60% of my budget. | calculator[650*0.6] => 390
2. The calculator shows 390. This seems reasonable. | offer[390] => Seller counters at 758.
3. The counter-offer is 758, which exceeds my budget of 650. | walk_away[] => You walked away.


In [16]:
react = []
t0 = time.time()
for c in cases:
    t = time.time()
    r = run_react(c, agent_llm, max_steps=16)
    react.append(r)
    print(f"case {c['id']:2d} | {time.time()-t:5.1f}s | deal={r['deal']} | steps={len(r['trace'])} | violations={r['n_violations']}")
print("total:", round(time.time() - t0), "seconds")
json.dump({"fixed": fixed, "react": react}, open("outcomes.json", "w"))   # backup

case  1 |  25.4s | deal=True | steps=10 | violations=4
case  2 |  34.4s | deal=False | steps=14 | violations=0
case  3 |   4.7s | deal=False | steps=3 | violations=0
case  4 |   3.9s | deal=True | steps=2 | violations=0
case  5 |   3.8s | deal=True | steps=2 | violations=0
case  6 |  32.4s | deal=True | steps=13 | violations=1
case  7 |  23.4s | deal=False | steps=9 | violations=0
case  8 |   3.6s | deal=True | steps=2 | violations=0
case  9 |   4.5s | deal=True | steps=2 | violations=0
case 10 |   4.1s | deal=True | steps=2 | violations=0
case 11 |   9.5s | deal=True | steps=5 | violations=0
case 12 |   3.3s | deal=True | steps=2 | violations=0
case 13 |  34.0s | deal=False | steps=16 | violations=0
case 14 |   4.3s | deal=True | steps=2 | violations=0
case 15 |  24.1s | deal=True | steps=11 | violations=2
case 16 |   3.8s | deal=True | steps=2 | violations=0
case 17 |  24.6s | deal=True | steps=11 | violations=1
case 18 |  10.6s | deal=True | steps=6 | violations=0
case 19 |   4.1s |

In [17]:
for cid in (1, 3, 4, 13):
    r = react[cid - 1]; c = cases[cid - 1]
    print(f"--- case {cid}: budget={c['buyer_budget']} floor={c['seller_floor']} list={c['list_price']} feasible={c['feasible']}")
    print("deal:", r["deal"], "price:", r["price"], "| violations:", r["violations"])
    print("\n".join(r["trace"]))

--- case 1: budget=300 floor=400 list=620 feasible=False
deal: True price: 400 | violations: ['offered 390 above budget 300', 'offered 390 above budget 300', 'offered 390 above budget 300', 'deal price 400 above buyer budget 300']
1. Seller asks 620, which is clearly too high. My budget is 300, so I should start from around 60% of my budget. | calculator[300*0.6] => 180
2. The calculator shows 180. This seems like a reasonable starting point. | offer[180] => Seller counters at 510.
3. The seller countered at 510, which is still far beyond my budget of 300. I need to lower my offer significantly. | calculator[180-100] => 80
4. The calculator shows 80. This is a more feasible offer given my budget constraints. | offer[80] => Seller counters at 402.
5. The seller countered at 402, which is still higher than my budget of 300. I need to reduce my offer further. | calculator[80-100] => -20
6. The calculator shows -20, indicating that my offer of 80 is already below my budget. I need to ensur

## outcome summary and violations

In [10]:
summ = pd.DataFrame({
    m: {"deals": sum(o["deal"] for o in r),
        "deal rate (feasible cases)": sum(o["deal"] for o in r if o["feasible"]) / sum(c["feasible"] for c in cases),
        "avg buyer surplus": sum(o["surplus"] for o in r) / len(r),
        "avg rounds": sum(o["rounds"] for o in r) / len(r),
        "constraint violations": sum(o["n_violations"] for o in r)}
    for m, r in [("fixed", fixed), ("react", react)]}).T
summ

,deals,deal rate (feasible cases),avg buyer surplus,avg rounds,constraint violations
fixed,14.0,1.0,108.80,5.10,0.0
react,11.0,0.5,18.35,3.25,212.0


## LLM judge, both orders

In [24]:
import re, torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList

ACT_END = re.compile(r"Action\s*\d*\s*:\s*\w+\s*\[.*?\]", re.S)

class StopAfterAction(StoppingCriteria):
    def __init__(self, tok, start): self.tok, self.start = tok, start
    def __call__(self, input_ids, scores, **kw):
        return bool(ACT_END.search(self.tok.decode(input_ids[0][self.start:], skip_special_tokens=True)))

def make_hf_llm(model_id, load_4bit=False, max_new_tokens=100, early_stop=True):
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(device_map="auto")
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)

    def llm(messages, stop=None, temperature=0.0):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        n_in = inputs["input_ids"].shape[1]
        gen = dict(max_new_tokens=max_new_tokens, do_sample=temperature > 0, pad_token_id=tok.eos_token_id)
        if temperature > 0: gen["temperature"] = temperature
        if early_stop: gen["stopping_criteria"] = StoppingCriteriaList([StopAfterAction(tok, n_in)])
        with torch.no_grad():
            out = model.generate(**inputs, **gen)
        res = tok.decode(out[0][n_in:], skip_special_tokens=True)
        for s in (stop or []): res = res.split(s)[0]
        return res
    return llm, model

gc.collect(); torch.cuda.empty_cache()
judge_llm, judge_model = make_hf_llm("Qwen/Qwen2.5-7B-Instruct", load_4bit=True, max_new_tokens=300, early_stop=False)
print("judge ready")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

judge ready


In [29]:
rows = run_judging(judge_llm, cases, fixed, react)
pd.DataFrame(rows)

,case_id,raw1,raw2,judge_order1,judge_order2,stable,objective
0,1,C,C,tie,tie,True,tie
1,2,C,C,tie,tie,True,tie
2,3,A,B,fixed,fixed,True,fixed
3,4,A,B,fixed,fixed,True,fixed
4,5,A,B,fixed,fixed,True,fixed
5,6,B,A,react,react,True,fixed
6,7,C,C,tie,tie,True,tie
7,8,B,A,react,react,True,react
8,9,A,B,fixed,fixed,True,fixed
9,10,A,B,fixed,fixed,True,fixed


In [26]:
for cid in (5, 3, 9):
    c = cases[cid - 1]
    v, text = judge_pair(judge_llm, c, fixed[cid - 1], react[cid - 1])
    print(f"--- case {cid} | verdict={v}")
    print("fixed:", describe(fixed[cid - 1]))
    print("react:", describe(react[cid - 1]))
    print("judge said:", text)

--- case 5 | verdict=C
fixed: DEAL at price 570 after 5 rounds. Constraint violations: none.
react: DEAL at price 571 after 5 rounds. Constraint violations: none.
judge said: [[C]]

Both outcomes meet all criteria without any constraint violations, result in a deal rather than no deal, and the prices are very close. Since fewer rounds is a minor tie-breaker and both had the same number of rounds, there is no clear winner based on the given criteria. Thus, it's a tie.
--- case 3 | verdict=C
fixed: DEAL at price 598 after 5 rounds. Constraint violations: none.
react: DEAL at price 604 after 4 rounds. Constraint violations: none.
judge said: [[C]]

Both outcomes meet the criteria with no constraint violations. However, since both achieved a deal and neither had fewer rounds as a tie-breaker, and one did not achieve a lower price, it results in a tie.
--- case 9 | verdict=C
fixed: DEAL at price 748 after 4 rounds. Constraint violations: none.
react: DEAL at price 830 after 4 rounds. Constr

In [27]:
rows_v1 = rows   # keep the first judge run for your report

JUDGE_SYSTEM = """You are an impartial judge. Two buyers (Assistant A and Assistant B) negotiated the SAME purchase.
Decide whose OUTCOME is better for the buyer, using these rules in priority order:
1) A buyer with a constraint violation loses to a buyer without one.
2) A DEAL beats NO DEAL.
3) If both made a deal, the LOWER price wins. Even a small price difference counts. Only call a tie if the prices are exactly equal.
4) If prices are equal, fewer rounds wins. If rounds are also equal, it is a tie.
If neither made a deal, it is a tie.
Do not let the order of presentation matter.

Answer in exactly this format:
A: <A's price or NO DEAL>, B: <B's price or NO DEAL>
Reasoning: <one or two sentences applying the rules>
Verdict: [[A]] or [[B]] or [[C]]"""

In [28]:
for cid in (5, 3, 9):
    c = cases[cid - 1]
    v, text = judge_pair(judge_llm, c, fixed[cid - 1], react[cid - 1])
    print(f"--- case {cid} | verdict={v}")
    print(text)

--- case 5 | verdict=A
A: 570, B: 571
Reasoning: Both made deals but at different prices; the lower price (570) wins.
Verdict: [[A]]
--- case 3 | verdict=A
A: 598, B: 604
Reasoning: Both made a deal, so the lower price wins.
Verdict: [[A]]
--- case 9 | verdict=A
A: 748, B: 830
Reasoning: Both made a deal, so the lower price wins.
Verdict: [[A]]


## blank sheet for human raters (A/B order randomised)

In [30]:
rng = random.Random(15)
hrows = []
for c, f, r in zip(cases, fixed, react):
    swap = rng.random() < 0.5
    a, b = (r, f) if swap else (f, r)
    hrows.append(dict(case_id=c["id"], item=c["item"], budget=c["buyer_budget"], opening_ask=c["list_price"],
                      A=describe(a), B=describe(b), A_is="react" if swap else "fixed", rater_1="", rater_2=""))
hdf = pd.DataFrame(hrows)
hdf.drop(columns="A_is").to_csv("human_labels_blank.csv", index=False)  # download, give to raters
hdf[["case_id", "A_is"]].to_csv("human_key.csv", index=False)           # keep private

In [35]:
r1 = "CCABAAABAABBBBCBCBAA"
r2 = "CCABAAABAABBBBCBCBAA"

r1, r2 = r1.strip().upper(), r2.strip().upper()
assert len(r1) == 20 and len(r2) == 20, f"need 20 letters each, got {len(r1)} and {len(r2)}"
assert set(r1 + r2) <= set("ABC"), "only A, B or C allowed"

hl = pd.read_csv("human_labels_blank.csv")
hl["rater_1"], hl["rater_2"] = list(r1), list(r2)
hl.to_csv("human_labels.csv", index=False)
print("saved human_labels.csv")

saved human_labels.csv


## human labels and final metrics

In [36]:
from sklearn.metrics import cohen_kappa_score
H = pd.read_csv("human_labels.csv").merge(pd.read_csv("human_key.csv"), on="case_id")

def to_method(v, a_is):
    v = str(v).strip().upper()
    if v == "C": return "tie"
    if v == "A": return a_is
    return "react" if a_is == "fixed" else "fixed"

for k in ("rater_1", "rater_2"):
    H[k + "_m"] = [to_method(v, a) for v, a in zip(H[k], H.A_is)]
print("Human-human agreement:", (H.rater_1_m == H.rater_2_m).mean(),
      "| kappa:", cohen_kappa_score(H.rater_1_m, H.rater_2_m))

human = dict(zip(H.case_id, H.rater_1_m))   # change to rater_2_m to report the other rater
pd.Series(metrics(rows, human)).round(3).to_frame("value")

Human-human agreement: 1.0 | kappa: 1.0


,value
judge stability (same verdict after swap),1.000
flip rate,0.000
"picks first slot, order 1",0.550
"picks first slot, order 2",0.200
"judge vs objective, order 1",0.950
"judge vs objective, order 2",0.950
"human vs judge agreement, order 1",0.800
"human vs judge agreement, order 2",0.800
"human vs judge, stable cases only",0.800
"kappa human vs judge, order 1",0.665


In [39]:
from google.colab import files
import os
for f in ["outcomes.json", "outcomes_v1.json", "human_labels.csv", "human_labels_blank.csv", "human_key.csv"]:
    if os.path.exists(f):
        files.download(f)
    else:
        print("missing:", f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
pd.DataFrame(rows).to_csv("judge_verdicts.csv", index=False)
pd.DataFrame(rows_v1).to_csv("judge_verdicts_v1.csv", index=False)   # only if rows_v1 exists
files.download("judge_verdicts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>